# CARec with cached item embeddings

- Load one embedding condition and the saved interaction splits.
- Product text and sampling belong in [data_exploration.ipynb](data_exploration.ipynb). Encoder inference belongs in [carec_encoder_comparison.ipynb](carec_encoder_comparison.ipynb).


## What CARec already implements

- Build a user/product graph from training interactions and aggregate neighbours with LightGCN.
- Phase 1 learns the user embedding table while keeping cached product vectors fixed and bypassing the MLP.
- Phase 2 freezes the user table and learns an MLP that transforms the product vectors. Graph propagation still runs.
- Alignment brings interacting users and products closer; uniformity prevents collapse. The text encoder stays frozen.


In [11]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
DATASET_ID = "electronics_carec_035_seed2023"
PREPARED_DIR = PROJECT_ROOT / "data/processed" / DATASET_ID
EMBEDDING_NAME = "all-MiniLM-L6-v2.npz"
EMBEDDING_PATH = PROJECT_ROOT / "artifacts/embeddings" / DATASET_ID / EMBEDDING_NAME


## Load item embeddings

- Keep each vector paired with its original product ID.
- Change `EMBEDDING_NAME` to select another cached condition.


In [12]:
with np.load(EMBEDDING_PATH, allow_pickle=False) as saved:
    embedding_item_ids = saved["item_ids"]
    item_embeddings = saved["embeddings"]
    embedding_model_id = str(saved["model_id"])
    embedding_prompt = str(saved["prompt"])

print(embedding_model_id)
print(f"Embeddings: {item_embeddings.shape}")
print(f"Prompt: {embedding_prompt!r}")


sentence-transformers/all-MiniLM-L6-v2
Embeddings: (34220, 384)
Prompt: ''


## Load interaction splits

- Keep user and product IDs as strings.
- Only `train` interactions may enter the graph. Validation, test, and cold interactions stay held out.


In [13]:
partitions = {}
for name in ("train", "valid", "test", "cold"):
    partitions[name] = pd.read_csv(
        PREPARED_DIR / f"{name}.tsv.gz",
        sep="\t",
        dtype={"user_id:token": str, "item_id:token": str},
        float_precision="round_trip",
    )
    print(f"{name}: {len(partitions[name]):,} interactions")

train = partitions["train"]
valid = partitions["valid"]
test = partitions["test"]
cold = partitions["cold"]


train: 452,606 interactions
valid: 14,853 interactions
test: 133,274 interactions
cold: 23,163 interactions


## 1. Prepare files that RecBole understands

- RecBole is the external library that CARec uses to handle recommendation datasets.
- It reads interaction files named `<dataset>.train.inter`, `<dataset>.valid.inter`, and so on.
- Write copies of our existing splits under `artifacts/recbole/`. Keep the saved split boundaries and repeated interactions.
- The `.item` file lists every product, including cold products. Vectors stay in the loaded NumPy matrix for now.
- Export only the two ID fields and timestamp. `source_row` is an audit column without a RecBole type suffix, so it stays out of these files.


In [14]:
RECBOLE_ROOT = PROJECT_ROOT / "artifacts/recbole"
RECBOLE_DIR = RECBOLE_ROOT / DATASET_ID
RECBOLE_DIR.mkdir(parents=True, exist_ok=True)

interaction_columns = ["user_id:token", "item_id:token", "timestamp:float"]

for split_name, interactions in partitions.items():
    split_path = RECBOLE_DIR / f"{DATASET_ID}.{split_name}.inter"
    recbole_interactions = interactions[interaction_columns]
    recbole_interactions.to_csv(split_path, sep="\t", index=False)

product_catalogue = pd.DataFrame({"item_id:token": embedding_item_ids})
product_catalogue.to_csv(
    RECBOLE_DIR / f"{DATASET_ID}.item",
    sep="\t",
    index=False,
)
print(f"RecBole input files: {RECBOLE_DIR}")

RecBole input files: /Users/matouskovar/FIT/NI-PML/artifacts/recbole/electronics_carec_035_seed2023


## 2. Import the dataset tools

- `Config` holds RecBole settings, such as file names and column names.
- `Dataset` reads those files and assigns integer IDs to users and products. [Library implementation](https://github.com/RUCAIBox/RecBole/blob/v1.1.1/recbole/data/dataset/dataset.py).
- `load_carec()` imports our pinned CARec code. We use its model class in the configuration; we do not instantiate the model here.


In [15]:
import sys

from benchmarks.carec import load_carec
from recbole.config import Config
from recbole.data.dataset import Dataset

carec = load_carec()


## 3. Tell RecBole which files and fields to use

- `data_path` is the parent directory. RecBole appends `DATASET_ID` itself.
- `benchmark_filename` tells RecBole to keep our four saved splits, rather than create a new split.
- `load_col` lists field names without the `:token` or `:float` suffix.
- Disable numeric normalization. Keep the values we prepared. Dataset preparation uses CPU; training device selection comes later.


In [16]:
dataset_settings = {
    "data_path": str(RECBOLE_ROOT),
    "benchmark_filename": ["train", "valid", "test", "cold"],
    "field_separator": "\t",
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "TIME_FIELD": "timestamp",
    "load_col": {
        "inter": ["user_id", "item_id", "timestamp"],
        "item": ["item_id"],
    },
    "normalize_all": False,
    "use_gpu": False,
}

config = Config(
    model=carec.CollabContex,
    dataset=DATASET_ID,
    config_dict=dataset_settings,
)


## 4. Create the dataset and inspect its product IDs

- `Dataset(config)` loads the files and builds a shared ID vocabulary for all four splits.
- `field2id_token["item_id"]` translates a RecBole integer ID back to the original product ID.
- RecBole reserves ID `0` for `[PAD]`. This is a placeholder, not a real product.
- This dataset still contains all four splits. Later, only its training split may be passed to the graph model.


In [19]:
recbole_dataset = Dataset(config)
recbole_product_ids = recbole_dataset.field2id_token["item_id"]

print(f"Products including padding: {recbole_dataset.item_num:,}")
print(f"ID 0: {recbole_product_ids[0]}")
print(f"First real product: {recbole_product_ids[1]}")

for split_name, row_count in zip(config["benchmark_filename"], recbole_dataset.file_size_list):
    print(f"{split_name}: {row_count:,} interactions")


Products including padding: 34,221
ID 0: [PAD]
First real product: B00C7NSIO8
train: 452,606 interactions
valid: 14,853 interactions
test: 133,274 interactions
cold: 23,163 interactions


/Users/matouskovar/FIT/NI-PML/.venv/lib/python3.11/site-packages/recbole/data/dataset/dataset.py:638: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/Users/matouskovar/FIT/NI-PML/.venv/lib/python3.11/site-packages/recbole/data/dataset/dataset.py:640: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on

## 5. Find each product's row in our saved embeddings

- Our cache has its own row order. Row `10` need not be RecBole product ID `10`.
- Build a dictionary from original product ID to its row in `item_embeddings`.
- `enumerate` gives us both the row number and the product ID.


In [18]:
embedding_row_by_product = {}
for embedding_row, product_id in enumerate(embedding_item_ids):
    embedding_row_by_product[product_id] = embedding_row

assert len(embedding_row_by_product) == len(embedding_item_ids), "Duplicate product IDs in cache"


## 6. Arrange vectors in RecBole order

- Allocate one row per RecBole product ID, including padding.
- Leave row `0` as zeros. For each real product, copy its cached vector into its RecBole row.
- The values stay unchanged. We only reorder rows and add the padding row.


In [21]:
embedding_dimension = item_embeddings.shape[1]
recbole_item_embeddings = np.zeros(
    (recbole_dataset.item_num, embedding_dimension),
    dtype=np.float32,
)

assert recbole_product_ids[0] == "[PAD]"
assert recbole_dataset.item_num == len(embedding_item_ids) + 1

for recbole_id in range(1, recbole_dataset.item_num):
    product_id = recbole_product_ids[recbole_id]
    embedding_row = embedding_row_by_product[product_id]
    recbole_item_embeddings[recbole_id] = item_embeddings[embedding_row]

print(f"Aligned embeddings: {recbole_item_embeddings.shape}")


Aligned embeddings: (34221, 384)


## 7. Inspect a few assignments

- Compare RecBole's integer ID, the original product ID, and the cache row.
- `same_vector` compares the copied vector with its source. Each displayed value should be `True`.
- For MiniLM, the aligned matrix should have 34,221 rows and 384 columns. The extra row is padding.


In [22]:
from IPython.display import display

preview_rows = []
for recbole_id in range(1, min(5, recbole_dataset.item_num)):
    product_id = recbole_product_ids[recbole_id]
    embedding_row = embedding_row_by_product[product_id]
    preview_rows.append({
        "recbole_id": recbole_id,
        "product_id": product_id,
        "cache_row": embedding_row,
        "same_vector": np.array_equal(
            recbole_item_embeddings[recbole_id], item_embeddings[embedding_row],
        ),
    })

display(pd.DataFrame(preview_rows))
print("Padding vector, first six values:", recbole_item_embeddings[0, :6])


,recbole_id,product_id,cache_row,same_vector
0,1,B00C7NSIO8,18133,True
1,2,B00AWRUIY4,17004,True
2,3,B005F778JO,12043,True
3,4,B009NB8WRU,16209,True


Padding vector, first six values: [0. 0. 0. 0. 0. 0.]
